# Level Breakout

Level Breakout (Horizontal S/R) \
Built on the dedicated **`engine.level_detector`** — the stateful horizontal-level detector: resistance / support / pullback levels seeded at confirmed pivots and tracked forward until *invalidated* (absolute / percent / ATR tolerance). \
This is a **different, richer** level source than `fractal_breakout`, which derives its S/R from the lighter `indicators.detect_swing_*` fractal pivots (see `fractal_breakout.ipynb`). \
It is the level-detector member of the `level_*` family — room to grow `level_bounce` (fade at the level) and `level_retest` (break-and-retest) on the shared `engine.strategies.level_base.LevelStrategyBase`.

__How the Level-Breakout Algorithm Determines Entry/Exit:__
- Detects horizontal S/R via `engine.level_detector.detect_all_levels` (resistance + support; optional pullback).
- Entry is **geometry-driven** (ignores the detector's S/R labels): at each bar, active levels are split by their position vs the prior close.
- Long Entry: close decisively clears an *overhead* level (`prev_close < level`, `close > level + buffer·ATR`).
- Short Entry: close breaks an *underlying* level (`prev_close > level`, `close < level − buffer·ATR`).
- Stop: **structural**, anchored on the broken level (`level ∓ mult·ATR`), seeded at entry; exit preset `structural_rr2` takes profit at 2R, stop-first.
- Native flip: exit when the close crosses back through any active level.
- Look-ahead free: a level is only usable from its confirmation bar (`start_idx + pivot_window`) and only until its causal invalidation bar.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Level Breakout

It's a direction flip, not different entry logic:
- level_breakout has one entry signal: breakout (close crosses a level-detector S/R level).
- level_breakout and level_breakout_inv are two separate classes — ride the breakout vs fade it.
- The _inv is the same signal traded in the opposite direction.

In [ ]:
# Import level-breakout strategy (built on engine.level_detector)
from engine.strategies import LevelBreakoutStrategy

In [ ]:
# Backtest level-breakout strategy
config = StrategyConfig()
strategy = LevelBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Level-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Inverse Level Breakout

In [ ]:
# Import inverse level-breakout strategy (fade the breakout)
from engine.strategies import InverseLevelBreakoutStrategy

In [ ]:
# Backtest inverse level-breakout strategy
config = StrategyConfig()
strategy = InverseLevelBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# inverse level-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()